(사전작업: 원본 파일 불러오기 > 용량 축소를 위해 필링된 파일 다운로드)

In [ ]:
import pandas as pd

file_path = r"C:\Users\rladp\Desktop\P혈관질환\HN07_24_ALL_MERGED.csv"
df = pd.read_csv(file_path)
# print(df.head())

In [ ]:
# age >= 19 필터링
df = df[df['age'] >= 19]
# CSV로 작업중인 폴더에 저장
df.to_csv("HN07~24_age_19over.csv", index=False)
print("저장 완료")

저장 완료


필터링 된 파일 불러오기

In [1]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\rladp\Desktop\P혈관질환\HN07~24_age_19over.csv"
df = pd.read_csv(file_path)

print(df.head())

C:\Users\rladp\AppData\Local\Temp\ipykernel_14600\1075895770.py:5: DtypeWarning: Columns (1,35,36,59,75,89,131,158,221,245,291,321,327,368,371,410,411,413,414,416,417,418,451,469,474,477,489,491,493,535,589,756,757,761,819,820,822,824,910,920,932,989,994,1063,1065,1077,1079,1081,1084,1088,1184,1185,1198,1204,1241,1245,1251,1254,1270,1274,1354,1367,1369,1377,1382,1383,1393,1396,1412,1418,1433,1434,1435,1501,1509,1524,1527,1537,1545,1650,1719,1771,2052,2090,2096,2112,2117,2162,2163,2169,2171,2172,2178,2180,2181,2187,2189,2190,2196,2228,2251,2252,2253,2254) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


         mod_d          ID    ID_fam    year  region  town_t  apt_t   psu  \
0  2025.12.30.  WA35203401  WA352034  2024.0     1.0     1.0    1.0  WA35   
1  2025.12.30.  WA35203402  WA352034  2024.0     1.0     1.0    1.0  WA35   
2  2025.12.30.  WA35220401  WA352204  2024.0     1.0     1.0    1.0  WA35   
3  2025.12.30.  WA35235701  WA352357  2024.0     1.0     1.0    1.0  WA35   
4  2025.12.30.  WA35239101  WA352391  2024.0     1.0     1.0    1.0  WA35   

   sex   age  ...  HE_hg  HE_cd  HE_mn  wt_itvnt  wt_exnt  HE_hp  LS_IMPC1  \
0  2.0  25.0  ...    NaN    NaN    NaN       NaN      NaN    NaN       NaN   
1  1.0  28.0  ...    NaN    NaN    NaN       NaN      NaN    NaN       NaN   
2  2.0  32.0  ...    NaN    NaN    NaN       NaN      NaN    NaN       NaN   
3  2.0  30.0  ...    NaN    NaN    NaN       NaN      NaN    NaN       NaN   
4  1.0  33.0  ...    NaN    NaN    NaN       NaN      NaN    NaN       NaN   

   LS_IMPC2  LS_IMPC3  LS_IMPC4  
0       NaN       NaN       NaN  


colab 사용 시 파일 불러오기 코드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

file_path = '/content/drive/MyDrive/CV/HN07~24_age_19over.csv'
df = pd.read_csv(file_path, low_memory=False)

In [39]:
#행, 열 제한 해제
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

간접흡연노출 칼럼 정의

In [2]:
smoke_cols = ["BS8_2", "BS9_2", "BS13"]

exposure_binary = df[smoke_cols].replace({
    1: 1,
    2: 0
})

df["간접흡연노출"] = exposure_binary.sum(axis=1)

all_undefined = df[smoke_cols].isin([8, 9]).all(axis=1)
df.loc[all_undefined, "간접흡연노출"] = np.nan


주중 수면시간 칼럼 생성 및 통합

In [3]:
# 둘 다 결측이 아닌 행의 개수
df[["BP16_1", "BP16_11"]].dropna().shape[0]
# df[["BP16_2", "BP16_21"]].dropna().shape[0]

0

In [4]:
# 1️⃣ 88, 99 → 결측 처리
sleep_cols_1 = ["BP16_11", "BP16_12", "BP16_13", "BP16_14"]
df[sleep_cols_1] = df[sleep_cols_1].replace({88: np.nan, 99: np.nan})
valid_counts = df[sleep_cols_1].notna().sum(axis=1)

partial_rows = df[valid_counts.between(1, 3)]
partial_df = df.loc[valid_counts.between(1, 3), sleep_cols_1]
partial_df.shape

(27, 4)

In [5]:
df.loc[valid_counts.between(1, 3), sleep_cols_1].reset_index()

,index,BP16_11,BP16_12,BP16_13,BP16_14
0,5120,23.0,0.0,5.0,NaN
1,18846,NaN,NaN,5.0,0.0
2,21418,23.0,0.0,NaN,NaN
3,21661,NaN,NaN,3.0,0.0
4,21769,1.0,0.0,NaN,NaN
5,21889,NaN,NaN,6.0,30.0
6,21914,NaN,NaN,4.0,30.0
7,22797,NaN,NaN,6.0,0.0
8,36450,NaN,NaN,7.0,0.0
9,40553,NaN,NaN,5.0,0.0


In [6]:
# 1️⃣ 88, 99 → 결측
sleep_cols_1 = ["BP16_11", "BP16_12", "BP16_13", "BP16_14"]
df[sleep_cols_1] = df[sleep_cols_1].replace({88: np.nan, 99: np.nan})

# 2️⃣ hour 기준 계산 가능 마스크
# BP16_11(취침 시간-시)과 BP16_13(기상 시간-시)이 둘 다 존재할 때만 True
valid_sleep = (
    df["BP16_11"].notna() &
    df["BP16_13"].notna()
)

# 3️⃣ 분 결측 → 0분 보정 (계산 가능한 행만)
df.loc[valid_sleep, "BP16_12"] = df.loc[valid_sleep, "BP16_12"].fillna(0)
df.loc[valid_sleep, "BP16_14"] = df.loc[valid_sleep, "BP16_14"].fillna(0)

# 4️⃣ 24시간제 보정
df.loc[valid_sleep, "BP16_11_1"] = np.where(
    df.loc[valid_sleep, "BP16_11"].between(1, 12),
    df.loc[valid_sleep, "BP16_11"] + 24,
    df.loc[valid_sleep, "BP16_11"]
)

df.loc[valid_sleep, "BP16_13_1"] = np.where(
    df.loc[valid_sleep, "BP16_13"].between(1, 12),
    df.loc[valid_sleep, "BP16_13"] + 24,
    df.loc[valid_sleep, "BP16_13"]
)

# 5️⃣ 분 단위 계산
bed_time = (
    df.loc[valid_sleep, "BP16_11_1"] * 60 +
    df.loc[valid_sleep, "BP16_12"]
)

get_time = (
    df.loc[valid_sleep, "BP16_13_1"] * 60 +
    df.loc[valid_sleep, "BP16_14"]
)

df.loc[valid_sleep, "BP16_1_계산"] = get_time - bed_time

# 6️⃣ 음수 보정
df.loc[df["BP16_1_계산"] < 0, "BP16_1_계산"] += 1440

# 7️⃣ 시간 단위
df["BP16_1_시간"] = (df["BP16_1_계산"] / 60).round(2)

# 8️⃣ 기존 값 결측 시만 대체
df["BP16_1"] = df["BP16_1"].fillna(df["BP16_1_시간"])
df["BP16_1"].head(20)

0      6.83
1      8.00
2      5.00
3      7.50
4      8.00
5      8.00
6      6.50
7     11.00
8      8.50
9      7.50
10     7.00
11     8.33
12     5.33
13     7.00
14     5.00
15     6.67
16     7.00
17     8.50
18     8.00
19     5.50
Name: BP16_1, dtype: float64

주말 수면 시간 칼럼 생성

In [7]:
# 1️⃣ 88, 99 → 결측 처리
sleep_cols_2 = ["BP16_21", "BP16_22", "BP16_23", "BP16_24"]
df[sleep_cols_2] = df[sleep_cols_2].replace({88: np.nan, 99: np.nan})

# 2️⃣ 계산 가능한 행 마스크 (hour 기준)
valid_sleep_2 = (
    df["BP16_21"].notna() &
    df["BP16_23"].notna()
)

# 3️⃣ 분 결측 → 0분 보정 (계산 가능한 행만)
df.loc[valid_sleep_2, "BP16_22"] = df.loc[valid_sleep_2, "BP16_22"].fillna(0)
df.loc[valid_sleep_2, "BP16_24"] = df.loc[valid_sleep_2, "BP16_24"].fillna(0)

# 4️⃣ 24시간제 보정
df.loc[valid_sleep_2, "BP16_21_1"] = np.where(
    df.loc[valid_sleep_2, "BP16_21"].between(1, 12),
    df.loc[valid_sleep_2, "BP16_21"] + 24,
    df.loc[valid_sleep_2, "BP16_21"]
)

df.loc[valid_sleep_2, "BP16_23_1"] = np.where(
    df.loc[valid_sleep_2, "BP16_23"].between(1, 12),
    df.loc[valid_sleep_2, "BP16_23"] + 24,
    df.loc[valid_sleep_2, "BP16_23"]
)

# 5️⃣ 분 단위 시각 변환
bed_time_2 = (
    df.loc[valid_sleep_2, "BP16_21_1"] * 60 +
    df.loc[valid_sleep_2, "BP16_22"]
)

get_time_2 = (
    df.loc[valid_sleep_2, "BP16_23_1"] * 60 +
    df.loc[valid_sleep_2, "BP16_24"]
)

# 6️⃣ 수면시간 계산
df.loc[valid_sleep_2, "BP16_2_계산"] = get_time_2 - bed_time_2

# 7️⃣ 음수 보정
df.loc[df["BP16_2_계산"] < 0, "BP16_2_계산"] += 1440

# 8️⃣ 시간 단위 변환
df["BP16_2_시간"] = (df["BP16_2_계산"] / 60).round(2)

# 9️⃣ 기존 값이 결측일 때만 대체
df["BP16_2"] = df["BP16_2"].fillna(df["BP16_2_시간"])
df["BP16_2"].head(20)


0     11.00
1     11.00
2      7.00
3     10.00
4      8.00
5      9.00
6      8.50
7     11.00
8      9.00
9     13.00
10     8.00
11     7.17
12     6.00
13     8.00
14     5.00
15     8.00
16     9.00
17     9.50
18     8.00
19     7.00
Name: BP16_2, dtype: float64

하루 평균 수면 시간 칼럼 생성

GAN 고려

In [8]:
df.loc[df[["BP16_1", "BP16_2"]].notna().sum(axis=1) == 1, ["BP16_1", "BP16_2"]]

,BP16_1,BP16_2
19360,9.0,NaN
19471,10.0,NaN
36628,NaN,5.5
38022,5.0,NaN
41374,NaN,10.0
42460,5.5,NaN
44805,5.0,NaN
45826,NaN,9.0
47785,11.5,NaN
49264,NaN,8.0


In [9]:
df = df[~(df[["BP16_1", "BP16_2"]].notna().sum(axis=1) == 1)].copy()

# 하루 평균 수면시간 계산 (주중 5일, 주말 2일 기준)
df["하루평균_수면시간"] = ((df["BP16_1"] * 5) + (df["BP16_2"] * 2)) / 7

# 소수 둘째 자리로 반올림
df["하루평균_수면시간"] = df["하루평균_수면시간"].round(2)
df["하루평균_수면시간"].head(20)

0      8.02
1      8.86
2      5.57
3      8.21
4      8.00
5      8.29
6      7.07
7     11.00
8      8.64
9      9.07
10     7.29
11     8.00
12     5.52
13     7.29
14     5.00
15     7.05
16     7.57
17     8.79
18     8.00
19     5.93
Name: 하루평균_수면시간, dtype: float64

In [10]:

# 1. 가중 평균 계산 (둘 다 값이 있을 때의 결과가 기본값이 됨)
# 둘 중 하나라도 NaN이면 결과는 NaN이 됩니다.
df["하루평균_수면시간"] = ((df["BP16_1"] * 5) + (df["BP16_2"] * 2)) / 7

# 2. 결측치 처리 로직 추가
# - 가중 평균이 NaN인 경우(즉, BP16_1이나 BP16_2 중 하나라도 결측인 경우) BP16_1 값으로 채우고, BP16_1도 없으면 BP16_2 값으로 채웁니다.
df["하루평균_수면시간"] = df["하루평균_수면시간"].fillna(df["BP16_1"]).fillna(df["BP16_2"])

# 3. 소수 둘째 자리로 반올림
df["하루평균_수면시간"] = df["하루평균_수면시간"].round(2)

# 결과 확인
print(df[["BP16_1", "BP16_2", "하루평균_수면시간"]].head(20))

    BP16_1  BP16_2  하루평균_수면시간
0     6.83   11.00       8.02
1     8.00   11.00       8.86
2     5.00    7.00       5.57
3     7.50   10.00       8.21
4     8.00    8.00       8.00
5     8.00    9.00       8.29
6     6.50    8.50       7.07
7    11.00   11.00      11.00
8     8.50    9.00       8.64
9     7.50   13.00       9.07
10    7.00    8.00       7.29
11    8.33    7.17       8.00
12    5.33    6.00       5.52
13    7.00    8.00       7.29
14    5.00    5.00       5.00
15    6.67    8.00       7.05
16    7.00    9.00       7.57
17    8.50    9.50       8.79
18    8.00    8.00       8.00
19    5.50    7.00       5.93


In [11]:
df[["BP8", "하루평균_수면시간"]].dropna().shape[0]

0

BP8(하루 평균 수면시간)과 하루평균_수면시간 통합 -> BP8 내부로 통합

In [12]:
df["BP8"] = df["BP8"].replace({88: np.nan, 99: np.nan})

df["BP8"] = df["BP8"].fillna(df["하루평균_수면시간"])
df["BP8"].head(20)

0      8.02
1      8.86
2      5.57
3      8.21
4      8.00
5      8.29
6      7.07
7     11.00
8      8.64
9      9.07
10     7.29
11     8.00
12     5.52
13     7.29
14     5.00
15     7.05
16     7.57
17     8.79
18     8.00
19     5.93
Name: BP8, dtype: float64

수면시간 갭 칼럼 생성

In [13]:
# 하나가 결측이면 결과도 결측
df["sleep_gap"] = (df["BP16_1"] - df["BP16_2"]).abs()

LQ_3EQL 일상활동 <- LQ_4HT 일하기

LQ_4EQL 통증/불편 <- LQ_2HT 통증

LQ_5EQL 불안/우울 <- LQ_5HT 우울


In [14]:
ht_map = {1: 1, 2: 2, 3: 2, 4: 3}


# 일상활동
df["LQ_3EQL"] = (
    df["LQ_3EQL"]
      .replace({8: np.nan, 9: np.nan})
      .fillna(df["LQ_4HT"].map(ht_map))
)

# 통증
df["LQ_4EQL"] = (
    df["LQ_4EQL"]
      .replace({8: np.nan, 9: np.nan})
      .fillna(df["LQ_2HT"].map(ht_map))
)

# 불안 우울
df["LQ_5EQL"] = (
    df["LQ_5EQL"]
      .replace({8: np.nan, 9: np.nan})
      .fillna(df["LQ_5HT"].map(ht_map))
)


가족력

In [15]:

# 질환별 원본 칼럼 매핑 (부, 모, 형제 순)
disease_map = {
    '고혈압' : ['HE_HPfh1', 'HE_HPfh2', 'HE_HPfh3'],
    '고지혈증': ['HE_HLfh1', 'HE_HLfh2', 'HE_HLfh3'],
    '허혈성심장질환': ['HE_IHDfh1', 'HE_IHDfh2', 'HE_IHDfh3'],
    '뇌졸중': ['HE_STRfh1', 'HE_STRfh2', 'HE_STRfh3'],
    '당뇨병': ['HE_DMfh1', 'HE_DMfh2', 'HE_DMfh3'],
    '암': ['HE_CNfh11', 'HE_CNfh21', 'HE_CNfh31'] # 암은 의사진단여부(11, 21, 31) 기준
}

for disease, cols in disease_map.items():
    # 1. 전처리: 8(비해당), 9(모름)를 NaN으로 변환
    df[cols] = df[cols].replace({8: np.nan, 9: np.nan})

    # 2. 유무 변수 생성 (Binary: 하나라도 1이면 1)
    df[f'{disease}가족력유무'] = df[cols].max(axis=1)

    # 3. 점수 변수 생성 (Score: 1인 개수 합산 0~3)
    df[f'{disease}가족력점수'] = df[cols].sum(axis=1, min_count=1)

# 결과 확인
new_cols = [c for c in df.columns if '가족력' in c]
print(df[new_cols].head(30))

    고혈압가족력유무  고혈압가족력점수  고지혈증가족력유무  고지혈증가족력점수  허혈성심장질환가족력유무  허혈성심장질환가족력점수  \
0        0.0       0.0        0.0        0.0           0.0           0.0   
1        0.0       0.0        0.0        0.0           0.0           0.0   
2        0.0       0.0        0.0        0.0           0.0           0.0   
3        0.0       0.0        0.0        0.0           0.0           0.0   
4        0.0       0.0        0.0        0.0           0.0           0.0   
5        0.0       0.0        0.0        0.0           1.0           1.0   
6        1.0       1.0        0.0        0.0           0.0           0.0   
7        0.0       0.0        0.0        0.0           0.0           0.0   
8        0.0       0.0        0.0        0.0           0.0           0.0   
9        0.0       0.0        1.0        1.0           0.0           0.0   
10       1.0       1.0        0.0        0.0           0.0           0.0   
11       1.0       2.0        0.0        0.0           0.0           0.0   
12       1.0

신체활동

In [16]:
def calculate_knhanes_met(df):
    """
    국건영 데이터를 바탕으로 영역별 MET-분/주 점수를 산출하는 함수
    """
    data = df.copy()

    # 1. 전처리
    target_cols = [
        'BE3_72', 'BE3_73', 'BE3_74', 'BE3_82', 'BE3_83', 'BE3_84', # 일
        'BE3_92', 'BE3_93', 'BE3_94',                               # 이동
        'BE3_76', 'BE3_77', 'BE3_78', 'BE3_86', 'BE3_87', 'BE3_88'  # 여가
    ]

    for col in target_cols:
        # 모름/무응답(9) 처리
        data[col] = data[col].replace([9], np.nan)
        # 비해당(8) 처리 -> 0분으로 간주
        data[col] = data[col].replace([8], 0)
        # NaN인 경우에도 계산을 위해 0으로 채울지 여부는 분석가의 판단 (여기서는 0으로 처리)
        data[col] = data[col].fillna(0)

    # 2. 영역별 MET 계산 (가독성을 위해 파트별로 분리)

    # [일 - Work] 고강도(8) & 중강도(4)
    work_high = data['BE3_72'] * (data['BE3_73'] * 60 + data['BE3_74']) * 8
    work_mod  = data['BE3_82'] * (data['BE3_83'] * 60 + data['BE3_84']) * 4
    data['MET_work'] = work_high + work_mod

    # [이동 - Transport] 중강도(4) 수준으로 간주
    trans_mod = data['BE3_92'] * (data['BE3_93'] * 60 + data['BE3_94']) * 4
    data['MET_trans'] = trans_mod

    # [여가 - Leisure] 고강도(8) & 중강도(4)
    leisure_high = data['BE3_76'] * (data['BE3_77'] * 60 + data['BE3_78']) * 8
    leisure_mod  = data['BE3_86'] * (data['BE3_87'] * 60 + data['BE3_88']) * 4
    data['MET_leisure'] = leisure_high + leisure_mod

    # 3. 총합 계산
    data['MET_total'] = data['MET_work'] + data['MET_trans'] + data['MET_leisure']

    return data[['MET_work', 'MET_trans', 'MET_leisure', 'MET_total']]



In [17]:
    # 함수 실행과 동시에 원본 df에 바로 저장하기
df[['MET_work', 'MET_trans', 'MET_leisure', 'MET_total']] = calculate_knhanes_met(df)

# 3. 결과 확인 (이제 'data'가 아니라 'df'로 확인하세요)
print(df[['MET_work', 'MET_trans', 'MET_leisure', 'MET_total']].head(30))


    MET_work  MET_trans  MET_leisure  MET_total
0        0.0      560.0       7320.0     7880.0
1        0.0      400.0       7200.0     7600.0
2        0.0     5040.0          0.0     5040.0
3        0.0      840.0       1360.0     2200.0
4        0.0      300.0          0.0      300.0
5        0.0        0.0          0.0        0.0
6        0.0      240.0          0.0      240.0
7        0.0      360.0          0.0      360.0
8     2400.0     1200.0       1200.0     4800.0
9        0.0      480.0        320.0      800.0
10       0.0        0.0       1800.0     1800.0
11       0.0        0.0          0.0        0.0
12       0.0     1680.0          0.0     1680.0
13       0.0        0.0          0.0        0.0
14     720.0      720.0          0.0     1440.0
15       0.0      480.0        240.0      720.0
16    1240.0      240.0       1600.0     3080.0
17       0.0      400.0       1680.0     2080.0
18       0.0      480.0          0.0      480.0
19       0.0      800.0          0.0    

근력운동

In [18]:
# 예시: BE5_1이 3(2일) 이상인 경우를 1로 설정
df['pa_st'] = (df['BE5_1'] >= 3).astype(int)

좌식시간

In [19]:
def calculate_sitting_time(df):
    """
    BE8_1(시간)과 BE8_2(분)를 활용해 하루 평균 앉아 있는 시간(분)을 계산합니다.
    """
    data = df.copy()

    # 1. 전처리: 모름/무응답(99)은 NaN으로, 비해당(88, 소아/청소년 등)은 0으로 치환
    # 분석 대상이 성인이라면 비해당인 경우 보통 좌식 시간이 0이거나 분석 제외 대상일 수 있습니다.
    cols = ['BE8_1', 'BE8_2']

    for col in cols:
        data[col] = data[col].replace([99], np.nan)  # 모름/무응답
        data[col] = data[col].replace([88], 0)       # 비해당
        data[col] = data[col].fillna(0)                     # 계산을 위해 남은 NaN을 0으로 처리

    # 2. 총 분(Minute) 단위로 환산
    # 공식: (시간 * 60) + 분
    data['sitting_time_total'] = (data['BE8_1'] * 60) + data['BE8_2']

    return data['sitting_time_total']

# --- 실제 적용 방법 ---
# 기존 데이터프레임 df에 'sitting_time' 칼럼으로 추가
df['sitting_time'] = calculate_sitting_time(df)

# 결과 확인
print(df[['BE8_1', 'BE8_2', 'sitting_time']].head())

   BE8_1  BE8_2  sitting_time
0    8.0    0.0         480.0
1    5.0    0.0         300.0
2   12.0   30.0         750.0
3   12.0    0.0         720.0
4    7.0    0.0         420.0


총에너지소모량(TDEE)

In [20]:
def calculate_bmr_tdee(df):
    """
    키, 몸무게, 나이, 성별 및 MET 점수를 이용하여 BMR과 TDEE를 산출
    """
    data = df.copy()

    # 1. 필수 칼럼 전처리 (결측치 및 특수 코드 처리)
    # 국건영 칼럼명 예시: HE_ht(키), HE_wt(몸무게), age(나이), sex(성별)
    # 성별 코드: 1(남성), 2(여성) 기준

    cols_to_fix = ['HE_ht', 'HE_wt', 'age']

def calculate_bmr_tdee(df):
    data = df.copy()

    # 1. 필수 칼럼 전처리 (불필요한 for문 삭제)
    # 이미 수치형이고 결측치 처리가 되어 있다면 바로 다음 단계로 넘어갑니다.

    # 2. 기초대사량(BMR) 계산
    # 남성: 10*체중 + 6.25*키 - 5*나이 + 5
    # 여성: 10*체중 + 6.25*키 - 5*나이 - 161

    data['BMR'] = np.where(
        data['sex'] == 1,
        (10 * data['HE_wt']) + (6.25 * data['HE_ht']) - (5 * data['age']) + 5,
        (10 * data['HE_wt']) + (6.25 * data['HE_ht']) - (5 * data['age']) - 161
    )

    # 3. 활동 계수(Activity Factor) 부여
    # 이전 단계에서 구한 'MET_total' 점수를 기준으로 분류
    # 기준 출처: 일반적인 건강 가이드라인 및 GPAQ 활동량 분류 기준 활용

    def get_activity_factor(met):
        if pd.isna(met): return 1.2 # 정보 없으면 최소 활동으로 가정
        if met < 600:    return 1.2 # 비활동적 (Sedentary)
        if met < 3000:   return 1.5 # 보통 활동 (Moderately Active)
        return 1.75                 # 매우 활동적 (Highly Active)

    data['activity_factor'] = data['MET_total'].apply(get_activity_factor)

    # 4. 총 에너지 소모량(TDEE) 계산
    data['TDEE'] = data['BMR'] * data['activity_factor']

    return data[['BMR', 'activity_factor', 'TDEE']]

# --- 실제 데이터프레임에 적용 ---
# df에 BMR, TDEE 칼럼 추가
bmr_tdee_results = calculate_bmr_tdee(df)
df[['BMR', 'activity_factor', 'TDEE']] = bmr_tdee_results

# 결과 확인
print(df[['HE_ht', 'HE_wt', 'age', 'MET_total', 'BMR', 'TDEE']].head())

   HE_ht  HE_wt   age  MET_total       BMR       TDEE
0  158.4   54.7  25.0     7880.0  1251.000  2189.2500
1  188.2  121.2  28.0     7600.0  2253.250  3943.1875
2  154.8   55.6  32.0     5040.0  1202.500  2104.3750
3  168.8   57.2  30.0     2200.0  1316.000  1974.0000
4  180.5   55.0  33.0      300.0  1518.125  1821.7500


In [21]:
# 내 몸이 쓰는 에너지(TDEE) 대비 실제 총 섭취 칼로리(N_EN)의 비율
df['energy_balance_ratio'] = df['N_EN'] / df['TDEE']

print('energy_balance_ratio' in df.columns) # True가 나와야 합니다.

True


영양소 칼럼 생성

In [22]:
def calculate_all_nutrient_metrics(df):
# 1.0보다 크면: 권장 비중 보다 많이 섭취
# 1.0보다 작으면: 권장 비중 보다 적게 섭취
    df['carb_ratio'] = (df['N_CHO'] * 4) / (df['TDEE'] * 0.55)
    df['fat_ratio'] = (df['N_FAT'] * 9) / (df['TDEE'] * 0.25)
    df['prot_ratio'] = (df['N_PROT'] * 4) / (df['TDEE'] * 0.20)


    # 2. 영양소 불균형 지수 (Nutrient Imbalance Index) 계산
    # 각 비율이 1.0(이상적)에서 벗어난 절대값의 합산
    # 이 값이 0에 가까울수록 신체 활동량 대비 완벽한 영양 균형을 의미함
    df['nutrient_imbalance'] = (
        abs(1 - df['carb_ratio']) +
        abs(1 - df['fat_ratio']) +
        abs(1 - df['prot_ratio'])
    )
    return df

# 함수 실행 (df 자체에 모든 칼럼이 저장)
df = calculate_all_nutrient_metrics(df)

# 결과 확인
print(df[['carb_ratio', 'fat_ratio', 'prot_ratio', 'nutrient_imbalance']].head())

   carb_ratio  fat_ratio  prot_ratio  nutrient_imbalance
0    0.689555   0.859678    1.122639            0.573406
1    0.368219   0.354137    0.484185            1.793459
2    0.600569   0.742444    0.414036            1.242951
3    0.876533   2.074658    1.437739            1.635864
4    0.893124   1.275381    1.043520            0.425778


In [23]:
# 탄수화물 섭취 에너지 대비 지방 섭취 에너지 비율
# 이 값이 높을수록 고지방 식단, 낮을수록 고탄수화물 식단임을 의미
df['carb_fat_balance'] = (df['N_FAT'] * 9) / (df['N_CHO'] * 4)

In [24]:
# N_SFA: 총 포화지방산 섭취량(g)
# TDEE: 앞서 계산한 총 에너지 소모량
# 기준: 전체 에너지의 7% 이내 섭취 권고

def add_sfa_feature(df):
    data = df.copy()

    # 포화지방산 섭취 권장 한도 대비 비율
    # 이 값이 1.0을 초과하면 권장량(7%)보다 많이 먹고 있다는 의미입니다.
    data['sfa_adequacy_ratio'] = (data['N_SFA'] * 9) / (data['TDEE'] * 0.07)

    # 분석의 편의를 위해 '과잉 섭취 여부' 이진 변수도 함께 생성
    data['sfa_over_intake'] = (data['sfa_adequacy_ratio'] > 1.0).astype(int)

    return data[['sfa_adequacy_ratio', 'sfa_over_intake']]

# 데이터프레임 적용
df[['sfa_adequacy_ratio', 'sfa_over_intake']] = add_sfa_feature(df)

# 결과 확인
print(df[['N_SFA', 'TDEE', 'sfa_adequacy_ratio', 'sfa_over_intake']].head(5))

       N_SFA       TDEE  sfa_adequacy_ratio  sfa_over_intake
0  19.823062  2189.2500            1.164179                1
1   9.042805  3943.1875            0.294849                0
2  13.370306  2104.3750            0.816888                0
3  33.791386  1974.0000            2.200915                1
4  28.408387  1821.7500            2.004944                1


In [25]:
# 식이섬유 밀도 및 충족 비율 생성
# 기준: 1,000kcal당 12g (한국인 영양소 섭취기준 권고치)

def add_fiber_features(df):
    data = df.copy()

    # 1. 식이섬유 밀도 (Fiber Density: g/1000kcal)
    # 에너지 섭취량이 0인 경우 ZeroDivisionError 방지를 위해 소량의 값 추가 혹은 처리
    data['fiber_density'] = (data['N_TDF'] / (data['N_EN'] + 1e-9)) * 1000

    # 2. 식이섬유 권장량 대비 충족 비율
    # 12g/1000kcal를 100%로 보았을 때의 비율
    data['fiber_adequacy_ratio'] = data['fiber_density'] / 12

    return data[['fiber_density', 'fiber_adequacy_ratio']]

# 데이터프레임 적용
df[['fiber_density', 'fiber_adequacy_ratio']] = add_fiber_features(df)

# 결과 확인
print(df[['N_TDF', 'N_EN', 'fiber_adequacy_ratio']].head())

       N_TDF         N_EN  fiber_adequacy_ratio
0  18.375571  1808.385042              0.846776
1  19.380350  1547.432997              1.043683
2  20.312733  1278.249519              1.324255
3  13.552429  2896.453340              0.389914
4  16.545499  1971.580612              0.699333


In [ ]:
# 변수 목록
cols = [
    "year", "HE_glu", "HE_chol", "sex", "age",

    "incm5", "ho_incm5", "edu", "occp", "cfam",
    "allownc", "marri_1", "D_1_1", "D_2_1", "DI1_dg",

    "DI2_dg", "DE1_dg", "DN1_dg", "DI3_dg", "DI4_dg",
    "DJ2_dg", "DC01_dg", "DJ4_dg", "DJ8_dg", "DM4_dg",

    "BP17_dg", "M_2_yr", "M_HL_3", "M_HL_4", "M_HL_9",
    "M_HL_10", "BH1", "LQ4_00", "LQ1_sb", "LQ_1EQL",

    "LQ_3EQL", "LQ_4HT", "LQ_4EQL", "LQ_1HT", "AC1_yr",
    "MH1_yr", "EC1_1", "EC_stt_1", "EC_stt_2", "EC_wht_0",

    "EC_wht_23", "EC_wht_5", "BO1_1", "BO1_3", "BD1_11",
    "BD2_14", "BD2_31", "BP16_1", "BP16_2", "BP16_11",

    "BP16_13", "BP16_21", "BP16_23", "BP1", "BS3_1",
    "BS6_2_1", "BS8_2", "BS9_2", "BS13", "BE3_92",

    "BE9", "LW_mt", "LW_wh",
    "HE_fh", "HE_BMI", "HE_wc", "GS_SP_DXA", "HE_COPD1",

    "BIA_TBW", "T_Q_DZ", "T_Q_CR", "E_CT", "LS_VEG2",
    "LS_FRUIT", "LK_LB_US", "LK_LB_IT", "LK_LB_EF", "N_DIET_WHY",

    "N_EN", "N_WAT_C", "N_FAT", "N_CHOL", "N_CHO", "N_TDF",
    "N_SUGAR", "N_NA", "LF_SAFE", "간접흡연노출", "BP8", "sleep_gap",

    '고혈압가족력유무', '고혈압가족력점수', '고지혈증가족력유무', '고지혈증가족력점수', '허혈성심장질환가족력유무', '허혈성심장질환가족력점수',
    '뇌졸중가족력유무', '뇌졸중가족력점수', '당뇨병가족력유무', '당뇨병가족력점수', '암가족력유무', '암가족력점수',

    "pa_aerobic", 'MET_work', 'MET_trans', 'MET_leisure', 'MET_total', 'pa_st', 'BE5_1', 'sitting_time', 'BMR', 'TDEE', 'energy_balance_ratio',
#영양섭취
    'carb_ratio', 'fat_ratio', 'prot_ratio', 'nutrient_imbalance', 'carb_fat_balance',
    'N_SFA', 'sfa_adequacy_ratio', 'sfa_over_intake', 'fiber_adequacy_ratio', 'HE_HbA1c', 'wt_tot', "N_WAT_C", "NF_WATER"

]

print(f"전체 행 개수: {df.shape[0]}")
print(f"사용 변수 개수: {len(cols)}개")


전체 행 개수: 111368
사용 변수 개수: 129개


In [34]:
# 결측 요약 테이블 생성
missing_summary = pd.DataFrame({
    "변수명": cols,
    "순수 결측 수": df[cols].isna().sum().values,
    "결측 비율(%)": df[cols].isna().mean().values * 100
})

# 결측 비율 높은 순 정렬
missing_summary = missing_summary.sort_values("결측 비율(%)", ascending=False)

# 결측 비율 높은 순 확인
# display(missing_summary.head(25))
# 결측 비율 낮은 순 확인
display(missing_summary.sort_values("결측 비율(%)", ascending=True).head(100))

,변수명,순수 결측 수,결측 비율(%)
0,year,0,0.000000
111,MET_total,0,0.000000
3,sex,0,0.000000
125,sfa_over_intake,0,0.000000
4,age,0,0.000000
...,...,...,...
126,fiber_adequacy_ratio,57084,51.257094
88,N_TDF,57084,51.257094
86,N_CHOL,57084,51.257094
52,BP16_1,59252,53.203793


In [35]:
# 1. 선처리, 수치형 변수 등  제외
제외_cols = [
    # 수치형
    'age', 'year',
    "HE_BMI", "HE_wc", "N_WAT_C", "NF_WATER",
    "N_FAT", "N_CHOL", "N_CHO", "N_TDF",
    "N_SUGAR", "N_NA",
    "HE_HbA1c", "HE_TG", "HE_HDL_st2",
    "HE_glu", "HE_LDL_drct", "HE_chol", 'N_EN', 'wt_tot'

    '간접흡연노출',

    # 수면
    "BP16_1", "BP16_11", "BP16_12", "BP16_13", "BP16_14", "BP16_2", "BP16_21", "BP16_22", "BP16_23", "BP16_24", "BP8", "sleep_gap",
    #가족력
    '고혈압가족력유무', '고혈압가족력점수', '고지혈증가족력유무', '고지혈증가족력점수', '허혈성심장질환가족력유무', '허혈성심장질환가족력점수',
    '뇌졸중가족력유무', '뇌졸중가족력점수', '당뇨병가족력유무', '당뇨병가족력점수', '암가족력유무', '암가족력점수',
    #신체활동
    'MET_work', 'MET_trans', 'MET_leisure', 'MET_total', 'sitting_time', 'BMR', 'TDEE', 'energy_balance_ratio',
#영양섭취
    'carb_ratio', 'fat_ratio', 'prot_ratio', 'nutrient_imbalance' 'carb_fat_balance',
    'N_SFA', 'sfa_adequacy_ratio', 'sfa_over_intake', 'N_TDF', 'N_EN', 'fiber_adequacy_ratio'

]

# 2. 처리 대상 컬럼
target_cols = [
    c for c in cols
    if c in df.columns and c not in 제외_cols
]

# 3. 컬럼 단위 처리
for col in target_cols:

    series = df[col]

    # 1️⃣ 8888 / 9999
    if series.isin([8888, 9999]).any():
        df[col] = series.replace({
            8888: "비해당",
            9999: np.nan
        })
        continue

    # 2️⃣ 888 / 999
    if series.isin([888, 999]).any():
        df[col] = series.replace({
            888: "비해당",
            999: np.nan
        })
        continue

    # 3️⃣ 88 / 99
    if series.isin([88, 99]).any():
        df[col] = series.replace({
            88: "비해당",
            99: np.nan
        })
        continue

    # 4️⃣ 8 / 9
    if series.isin([8, 9]).any():
        df[col] = series.replace({
            8: "비해당",
            9: np.nan
        })


In [36]:
# 결측 요약 테이블 생성
missing_summary = pd.DataFrame({
    "변수명": cols,
    "순수 결측 수": df[cols].isna().sum().values,
    "결측 비율(%)": df[cols].isna().mean().values * 100
})

# 결측 비율 높은 순 정렬
missing_summary = missing_summary.sort_values("결측 비율(%)", ascending=False)
# 결측 비율 높은 순 확인
display(missing_summary.head(110))


,변수명,순수 결측 수,결측 비율(%)
71,GS_SP_DXA,109679,98.483406
72,HE_COPD1,108277,97.224517
21,DC01_dg,105876,95.068601
27,M_HL_3,99858,89.664895
28,M_HL_4,99850,89.657711
...,...,...,...
59,BS3_1,7456,6.694921
51,BD2_31,7310,6.563824
16,DE1_dg,6857,6.157065
15,DI2_dg,6847,6.148086


In [37]:
# 중복된 이름이 있는지 확인
from collections import Counter
counts = Counter(cols)
duplicates = [item for item, count in counts.items() if count > 1]
print("중복된 컬럼명:", duplicates)

중복된 컬럼명: []


In [38]:
# 유지할 변수
force_keep_cols = ['LS_FRUIT', 'LS_VEG2' ]

# 결측률 계산
missing_ratio = df[cols].isna().mean() * 100

# 결측 60% 미만 + 강제 유지
keep_cols = [
    c for c in cols
    if (missing_ratio[c] < 60) or (c in force_keep_cols)
]

# 새로운 DataFrame
new_df = df[keep_cols].copy()

# 결과 확인
print(f"원본: {len(cols)} | 제거: {len(cols)-len(keep_cols)} | 유지: {len(keep_cols)}")
print(f"new_df shape: {new_df.shape}")
print("제거된 컬럼:", [c for c in cols if c not in keep_cols])

원본: 129 | 제거: 19 | 유지: 110
new_df shape: (111368, 110)
제거된 컬럼: ['DC01_dg', 'BP17_dg', 'M_HL_3', 'M_HL_4', 'M_HL_9', 'M_HL_10', 'LQ_4HT', 'LQ_1HT', 'BP16_11', 'BP16_13', 'BP16_21', 'BP16_23', 'LW_wh', 'GS_SP_DXA', 'HE_COPD1', 'BIA_TBW', 'T_Q_DZ', 'T_Q_CR', 'E_CT']


In [39]:
# 결측 요약 테이블 생성
new_missing_summary = pd.DataFrame({
    "변수명": keep_cols,
    "순수 결측 수": new_df.isna().sum().values,
    "결측 비율(%)": new_df.isna().mean().values * 100
})

# 결측 비율 높은 순 정렬
new_missing_summary = new_missing_summary.sort_values("결측 비율(%)", ascending=False)
display(new_missing_summary.head(20))

,변수명,순수 결측 수,결측 비율(%)
59,LS_FRUIT,79216,71.129948
58,LS_VEG2,79216,71.129948
42,BD2_14,64533,57.945729
54,LW_mt,64049,57.511134
53,BE9,63796,57.283959
70,N_SUGAR,62390,56.021478
31,AC1_yr,61291,55.034660
32,MH1_yr,60389,54.224732
24,M_2_yr,60356,54.195101
75,sleep_gap,59258,53.209180


In [39]:
print("남은 변수 목록:", new_df.columns.tolist())
print(f"총 변수 개수: {len(new_df.columns)}개")

남은 변수 목록: ['year', 'HE_glu', 'HE_chol', 'sex', 'age', 'incm5', 'ho_incm5', 'edu', 'occp', 'cfam', 'allownc', 'marri_1', 'D_1_1', 'D_2_1', 'DI1_dg', 'DI2_dg', 'DE1_dg', 'DN1_dg', 'DI3_dg', 'DI4_dg', 'DJ2_dg', 'DJ4_dg', 'DJ8_dg', 'DM4_dg', 'M_2_yr', 'BH1', 'LQ4_00', 'LQ1_sb', 'LQ_1EQL', 'LQ_3EQL', 'LQ_4EQL', 'AC1_yr', 'MH1_yr', 'EC1_1', 'EC_stt_1', 'EC_stt_2', 'EC_wht_0', 'EC_wht_23', 'EC_wht_5', 'BO1_1', 'BO1_3', 'BD1_11', 'BD2_14', 'BD2_31', 'BP16_1', 'BP16_2', 'BP1', 'BS3_1', 'BS6_2_1', 'BS8_2', 'BS9_2', 'BS13', 'BE3_92', 'BE9', 'LW_mt', 'HE_fh', 'HE_BMI', 'HE_wc', 'LS_VEG2', 'LS_FRUIT', 'LK_LB_US', 'LK_LB_IT', 'LK_LB_EF', 'N_DIET_WHY', 'N_EN', 'N_WAT_C', 'N_FAT', 'N_CHOL', 'N_CHO', 'N_TDF', 'N_SUGAR', 'N_NA', 'LF_SAFE', '간접흡연노출', 'BP8', 'sleep_gap', '고혈압가족력유무', '고혈압가족력점수', '고지혈증가족력유무', '고지혈증가족력점수', '허혈성심장질환가족력유무', '허혈성심장질환가족력점수', '뇌졸중가족력유무', '뇌졸중가족력점수', '당뇨병가족력유무', '당뇨병가족력점수', '암가족력유무', '암가족력점수', 'pa_aerobic', 'MET_work', 'MET_trans', 'MET_leisure', 'MET_total', 'pa_st', 'BE5_1', 'si

In [ ]:
# colab version
from google.colab import files

new_df.to_csv('new_df.csv', index=False, encoding='utf-8-sig')
files.download('new_df.csv')

print("✅ 다운로드 시작! 브라우저 다운로드 폴더에 저장됩니다.")

In [ ]:
# vscode
# 원하는 경로에 저장
path = r'C:\Users\rladp\Desktop\P혈관질환\processed_HN07~24.csv(1)'
new_df.to_csv(path, index=False, encoding='utf-8-sig')